# Giai đoạn 3 - Sinh và kiểm tra patch

MiMo và OpenAI đọc fixture rồi gọi tool `propose_patch`. Patch chỉ được áp dụng trong thư mục tạm để chạy `git apply --check` và `pytest`; workspace thật không thay đổi.

Gemma4 Local được đánh dấu `SKIPPED` vì chưa vượt qua agent read-only nhiều bước ở Giai đoạn 2.

In [ ]:
# Cell 1: Dependency
%pip install -q requests python-dotenv pytest

In [ ]:
# ============================================================
# Cell 2: Cấu hình tools, validator và model
# ============================================================
import json
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import requests
from dotenv import load_dotenv


def find_repo_env():
    for directory in [Path.cwd(), *Path.cwd().parents]:
        candidate = directory / ".env"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy .env")


ENV_PATH = find_repo_env()
REPO_ROOT = ENV_PATH.parent
TEST_ROOT = REPO_ROOT / "Notebooks" / "Test_Code_Editor"
WORKSPACE = TEST_ROOT / "fixtures" / "sample_project"
RESULTS_DIR = TEST_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(TEST_ROOT) not in sys.path:
    sys.path.insert(0, str(TEST_ROOT))

from patch_validator import PatchValidator
from read_only_tools import ReadOnlyToolbox, tool_result_json

load_dotenv(ENV_PATH, override=False)
LITELLM_URL = os.getenv("LITELLM_URL", "http://localhost:4000/v1").rstrip("/")
LITELLM_MASTER_KEY = os.getenv("LITELLM_MASTER_KEY", "sk-local")
HEADERS = {
    "Authorization": f"Bearer {LITELLM_MASTER_KEY}",
    "Content-Type": "application/json",
}

MODEL_CASES = {
    "mimo": "mimo-pro",
    "openai": "openai-model",
}
SKIPPED_MODELS = {
    "local": {
        "model": "local-gemma",
        "reason": "Failed Stage 2 multi-step read-only agent gate",
    }
}
MAX_ITERATIONS = 8

read_tools = ReadOnlyToolbox(WORKSPACE, REPO_ROOT)
validator = PatchValidator(WORKSPACE)
TOOLS = read_tools.tool_schemas() + [validator.tool_schema()]

print(f"Workspace: {WORKSPACE}")
print(f"Models:    {MODEL_CASES}")
print(f"Skipped:   {SKIPPED_MODELS}")

In [ ]:
# ============================================================
# Cell 3: Agent loop với propose_patch dry-run
# ============================================================
SYSTEM_PROMPT = """Bạn là Coding Agent ở chế độ PROPOSE_ONLY.
Bạn được đọc project và đề xuất đúng một unified diff, nhưng không được ghi file.
Bắt buộc đọc calculator.py và tests/test_calculator.py trước khi đề xuất.
Patch phải là unified diff chuẩn gồm các dòng:
diff --git a/calculator.py b/calculator.py
--- a/calculator.py
+++ b/calculator.py
@@ ... @@
Chỉ sửa calculator.py, không sửa test hoặc README.
Mục tiêu là raise ValueError với message chứa 'must not be zero'.
Khi đã có patch, gọi propose_patch. Không in patch như text thay cho tool call."""

TASK = """Đọc calculator.py và tests/test_calculator.py.
Đề xuất patch tối thiểu để test chia cho 0 pass.
Không áp dụng patch vào workspace thật."""


def parse_arguments(raw):
    if isinstance(raw, dict):
        return raw
    return json.loads(raw or "{}")


def run_patch_agent(model_alias):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": TASK},
    ]
    trace = []
    read_paths = set()
    proposal_result = None
    proposal_arguments = None
    started = time.perf_counter()

    for iteration in range(1, MAX_ITERATIONS + 1):
        response = requests.post(
            f"{LITELLM_URL}/chat/completions",
            headers=HEADERS,
            json={
                "model": model_alias,
                "messages": messages,
                "tools": TOOLS,
                "tool_choice": "auto",
                "temperature": 0,
                "max_tokens": 1800,
            },
            timeout=180,
        )
        response.raise_for_status()
        data = response.json()
        choice = data.get("choices", [{}])[0]
        message = choice.get("message", {}) or {}
        tool_calls = message.get("tool_calls") or []
        messages.append(message)

        step = {
            "iteration": iteration,
            "actual_model": data.get("model"),
            "finish_reason": choice.get("finish_reason"),
            "assistant_content": message.get("content") or "",
            "usage": data.get("usage", {}),
            "tool_calls": [],
        }

        if not tool_calls:
            trace.append(step)
            missing = {"calculator.py", "tests/test_calculator.py"} - read_paths
            if missing or proposal_result is None:
                messages.append(
                    {
                        "role": "user",
                        "content": (
                            f"Chưa hoàn thành. File chưa đọc: {sorted(missing)}. "
                            f"Đã gọi propose_patch: {proposal_result is not None}. "
                            "Hãy tiếp tục dùng tool."
                        ),
                    }
                )
                continue
            return {
                "ok": True,
                "model": model_alias,
                "answer": message.get("content") or "",
                "trace": trace,
                "proposal": proposal_arguments,
                "validation": proposal_result,
                "iterations": iteration,
                "latency_seconds": round(time.perf_counter() - started, 3),
            }

        for tool_call in tool_calls:
            function = tool_call.get("function", {}) or {}
            tool_name = function.get("name", "")
            try:
                arguments = parse_arguments(function.get("arguments"))
            except (TypeError, json.JSONDecodeError) as error:
                arguments = {}
                result = {"ok": False, "error": f"Invalid arguments: {error}"}
            else:
                if tool_name == "propose_patch":
                    proposal_arguments = arguments
                    proposal_result = validator.safe_validate(arguments)
                    result = proposal_result
                else:
                    result = read_tools.execute(tool_name, arguments)
                    if tool_name == "read_file" and result.get("ok"):
                        file_path = arguments.get("file_path")
                        if isinstance(file_path, str):
                            read_paths.add(file_path)

            step["tool_calls"].append(
                {
                    "name": tool_name,
                    "arguments": arguments,
                    "result": result,
                }
            )
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.get("id"),
                    "content": tool_result_json(result),
                }
            )
        trace.append(step)

    return {
        "ok": False,
        "model": model_alias,
        "error": f"Exceeded {MAX_ITERATIONS} iterations",
        "trace": trace,
        "proposal": proposal_arguments,
        "validation": proposal_result,
        "latency_seconds": round(time.perf_counter() - started, 3),
    }

In [ ]:
# ============================================================
# Cell 4: Chấm patch
# ============================================================

def evaluate_result(result, snapshot_before, snapshot_after):
    validation = result.get("validation") or {}
    validated = validation.get("result", {}) if validation.get("ok") else {}
    patch = validated.get("normalized_patch", "")
    patched_content = validated.get("patched_content", "")
    test_result = validated.get("test_result", {})
    tool_names = [
        call["name"]
        for step in result.get("trace", [])
        for call in step.get("tool_calls", [])
    ]
    checks = {
        "agent_completed": result.get("ok") is True,
        "read_calculator": any(
            call["name"] == "read_file"
            and call["arguments"].get("file_path") == "calculator.py"
            for step in result.get("trace", [])
            for call in step.get("tool_calls", [])
        ),
        "read_test_file": any(
            call["name"] == "read_file"
            and call["arguments"].get("file_path") == "tests/test_calculator.py"
            for step in result.get("trace", [])
            for call in step.get("tool_calls", [])
        ),
        "called_propose_patch": "propose_patch" in tool_names,
        "patch_valid": validation.get("ok") is True,
        "only_calculator_changed": validated.get("changed_paths") == ["calculator.py"],
        "adds_zero_check": "if b == 0" in patched_content,
        "raises_expected_value_error": (
            "ValueError" in patched_content and "must not be zero" in patched_content
        ),
        "temporary_tests_pass": test_result.get("passed") is True,
        "real_workspace_unchanged": snapshot_before == snapshot_after,
        "no_test_or_readme_patch": (
            "tests/test_calculator.py" not in patch and "README.md" not in patch
        ),
    }
    return checks

In [ ]:
# ============================================================
# Cell 5: Chạy MiMo và OpenAI
# ============================================================
stage3_results = {}

for label, model_alias in MODEL_CASES.items():
    print(f"\n{'=' * 24} {label.upper()} {'=' * 24}")
    before = validator.workspace_snapshot()
    try:
        result = run_patch_agent(model_alias)
    except requests.RequestException as error:
        response = getattr(error, "response", None)
        result = {
            "ok": False,
            "model": model_alias,
            "error": response.text[:3000] if response is not None else str(error),
            "trace": [],
        }
    after = validator.workspace_snapshot()
    checks = evaluate_result(result, before, after)
    stage3_results[label] = {
        "provider": label,
        "model": model_alias,
        "tested_at": datetime.now(timezone.utc).isoformat(),
        "result": result,
        "checks": checks,
        "passed": all(checks.values()),
    }
    print(f"Checks: {checks}")
    print(f"Passed: {stage3_results[label]['passed']}")
    proposal = result.get("proposal") or {}
    if proposal:
        print(f"Explanation: {proposal.get('explanation')}")
        print(f"Patch:\n{proposal.get('patch')}")
    if result.get("error"):
        print(f"Error: {result['error']}")

In [ ]:
# ============================================================
# Cell 6: Lưu kết quả
# ============================================================
summary = [
    {
        "provider": "local",
        "model": "local-gemma",
        "status": "SKIPPED",
        "passed": False,
        "reason": SKIPPED_MODELS["local"]["reason"],
    }
]

for label, data in stage3_results.items():
    output = RESULTS_DIR / f"stage3_{label}.json"
    output.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    result = data["result"]
    summary.append(
        {
            "provider": label,
            "model": data["model"],
            "status": "PASS" if data["passed"] else "FAIL",
            "passed": data["passed"],
            "iterations": result.get("iterations"),
            "latency_seconds": result.get("latency_seconds"),
        }
    )
    print(f"Đã lưu {output}")

(RESULTS_DIR / "stage3_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n" + "=" * 70)
for item in summary:
    print(
        f"{item['provider']:<10} {item['model']:<18} "
        f"{item['status']:<8} {str(item.get('latency_seconds', '-')):>8}"
    )

assert validator.workspace_snapshot() == before or all(
    data["checks"]["real_workspace_unchanged"]
    for data in stage3_results.values()
), "Real workspace changed during Stage 3"

## Tiêu chí đạt

- Đọc cả code và test.
- Gọi `propose_patch` với unified diff hợp lệ.
- Chỉ sửa `calculator.py`.
- Thêm kiểm tra `b == 0` và `ValueError` đúng message.
- `pytest` pass trong bản sao tạm.
- Hash workspace thật không thay đổi.